In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
from cyclical_MCMC import load_data
from Heston import full_trunc_euler, moment_vec, SMM

# 1. Load Data

In [4]:
Y, dY, dY_pd, Y_indx, _ = load_data('data/SP500.csv')

Data from 2016-09-19 00:00:00 to 2026-09-17 00:00:00


# 2. Estimate with Simulated Method of Moments

### Settings for simulation

In [5]:
years = 10
days_pr_year = 252
T = years*days_pr_year # total number of days
burnin = days_pr_year # one year for burnin
N = 1000
Delta = 1

In [6]:
K_y = 4
L = 10
K_rv = 10
K_yrv = 5

M_data = moment_vec(dY, K_y, L, K_rv, K_yrv)

In [7]:
seed = 42
W_start = np.diag(1/(M_data**2+1e-3))
N_multi = 50
iter = 20
W_tol = 1e-8

# bounds for optimization
lb = [0, 0, 0.1, 0.005, -0.99]
ub = [0.2, 0.1, 2.5, 0.15, 0.99]

In [8]:
lens_roll = N_multi*[10]

# bounds for drawing initial guesses
bounds = {
    'mu_l' : 0.02, 'mu_h' : 0.07,
    'kappa_l' : 0.01, 'kappa_h' : 0.1,
    'theta_l' : 0.5, 'theta_h' : 2,
    'sigma_sq_V_l' : 0.01, 'sigma_sq_V_h' : 0.06,
    'rho_l' : -0.8, 'rho_h' : -0.2
}

In [9]:
out = SMM(
    dY, N_multi, iter, W_start, lb, ub,
    K_y, L, K_rv, K_yrv,
    seed, bounds, lens_roll, Delta, N, T, burnin, W_tol
)

----------------------------------------------------------------------------------------------------
Starting Multistart i = 0 with guess:
mu = 0.02741
kappa = 0.02452
theta = 0.96814
sigma_sq_V = 0.05965
rho = -0.51668
Ended after 25 iterations with status = 3
Theta:
mu = 0.04427
kappa = 0.01098
theta = 1.38579
sigma_sq_V = 0.14997
rho = -0.99000
cost = 1.01781618

----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------
Starting Multistart i = 1 with guess:
mu = 0.04397
kappa = 0.07370
theta = 1.27735
sigma_sq_V = 0.05207
rho = -0.44176
Ended after 29 iterations with status = 3
Theta:
mu = 0.04406
kappa = 0.01091
theta = 1.38092
sigma_sq_V = 0.14730
rho = -0.98985
cost = 1.03342292

----------------------------------------------------------------------------------------------------
------------------------------------------------------------

In [ ]:
full_trunc_euler()

<function Heston.full_trunc_euler(mu, kappa, theta, sigma_sq_V, rho, Delta, N=1, T=1, burnin=0, do_print=False)>

array([ 0.06320323,  0.02636154,  0.87006892,  0.039174  , -0.99      ])